In [14]:
candidates = [
    {
        "id": "cv-001",
        "full_name": "Nguyen Van An",
        "current_job_title": "Senior Software Engineer",
        "education_text": "BS Computer Science, MS Software Engineering",
        "experience_text": "4 years at TechCorp, 1 year at InnovateX, backend development",
        "skills_text": "Python, SQL, Machine Learning, FastAPI, Docker",
        "summary_text": "5 years backend experience, scalable systems, CI/CD",
    },
    {
        "id": "cv-002",
        "full_name": "Tran Thi B",
        "current_job_title": "Marketing Specialist",
        "education_text": "BA Business Administration",
        "experience_text": "3 years in digital marketing and campaign analytics",
        "skills_text": "CRM, communication, Google Ads",
        "summary_text": "Marketing-focused profile",
    },
    {
        "id": "cv-003",
        "full_name": "Le Van C",
        "current_job_title": "Data Analyst",
        "education_text": "BS Statistics, MS Data Science",
        "experience_text": "2 years at DataInsights, 1 year at AnalyticsPro, data visualization and analysis",
        "skills_text": "R, Tableau, SQL, Python",
        "summary_text": "Data analysis and visualization expertise",
    }
]

def router_node(state):
    # Call LLM to decide route
    return {
        "current_CVs": [candidate["id"] for candidate in candidates],
        "dsl_query": "Find candidates with marketing experience and skills",
        "llm_query": "Find the best candidate for a marketing role based on CV summaries",
        "confidence": 0.8,
        "reason": "Objective is to extract structured data, DSL is more reliable for this task",
        }

def dsl_node(state):
    # Call DSL tool
    return {"dsl_result": ["cv-001", "cv-002", "cv-003"]}


def llm_node(state):
    # Call LLM tool
    return {"llm_result": ["cv-002"]}

def hybrid_node(state):
    # Combine results
    dsl_candidates = set(state.get("dsl_result", []))
    llm_candidates = set(state.get("llm_result", []))
    final_candidates = dsl_candidates.intersection(llm_candidates)
    return {"final_candidates": list(final_candidates)}

In [15]:
import json
import re
import sys
from pathlib import Path
from typing import TypedDict, List, Literal, Dict, Any

from langgraph.graph import StateGraph, START, END

# Ensure project root is importable when notebook runs from notebooks/.
project_root = Path.cwd()
if project_root.name.lower() == "notebooks":
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from backend.src.services.ai_agent.providers import LLMProvider, LLMProviderError


class DemoState(TypedDict, total=False):
    user_question: str
    route: Literal["dsl", "llm", "hybrid"]
    reason: str
    router_raw: str
    router_provider: str
    dsl_result: List[str]
    llm_result: List[str]
    final_candidates: List[str]
    final_payload: Dict[str, Any]
    answer: str


def _safe_json_extract(text: str):
    text = (text or "").strip()
    if not text:
        return None

    try:
        return json.loads(text)
    except Exception:
        pass

    match = re.search(r"\{[\s\S]*\}", text)
    if match:
        try:
            return json.loads(match.group(0))
        except Exception:
            return None
    return None


def _heuristic_route(question: str):
    q = (question or "").lower()
    if any(k in q for k in ["kết hợp", "ket hop", "combine", "both", "hybrid"]):
        return "hybrid", "Fallback heuristic: user asks to combine sources."
    if any(k in q for k in ["summary", "tóm tắt", "tom tat", "nhận định", "nhan dinh", "đánh giá tổng quan", "danh gia tong quan"]):
        return "llm", "Fallback heuristic: natural-language reasoning request."
    return "dsl", "Fallback heuristic: structured filtering intent."


def route_node(state: DemoState):
    question = state.get("user_question", "")

    system_prompt = (
        "You are a routing model. Choose exactly one route for recruitment query handling: "
        "dsl, llm, or hybrid. Return JSON only with keys: route, reason."
    )
    user_prompt = (
        f"Question: {question}\n"
        "Rules:\n"
        "- route=dsl for structured filtering/constraints\n"
        "- route=llm for subjective reasoning/summarization\n"
        "- route=hybrid when user explicitly wants combination or both\n"
        "Output JSON only."
    )

    try:
        provider = LLMProvider()
        resp = provider.generate(prompt=user_prompt, system_prompt=system_prompt)
        parsed = _safe_json_extract(resp.text)

        if parsed and parsed.get("route") in {"dsl", "llm", "hybrid"}:
            return {
                "route": parsed["route"],
                "reason": str(parsed.get("reason", "LLM router decision")),
                "router_raw": resp.text,
                "router_provider": resp.provider,
            }

        route, reason = _heuristic_route(question)
        return {
            "route": route,
            "reason": f"LLM output parse failed. {reason}",
            "router_raw": resp.text,
            "router_provider": resp.provider,
        }

    except (LLMProviderError, Exception) as exc:
        route, reason = _heuristic_route(question)
        return {
            "route": route,
            "reason": f"LLM unavailable ({exc}). {reason}",
            "router_raw": "",
            "router_provider": "fallback",
        }


def route_selector(state: DemoState) -> Literal["dsl", "llm", "hybrid"]:
    return state["route"]


def hybrid_exec_node(state: DemoState):
    dsl_out = dsl_node(state)
    llm_out = llm_node(state)
    merged_state = {**state, **dsl_out, **llm_out}
    hybrid_out = hybrid_node(merged_state)
    return {**dsl_out, **llm_out, **hybrid_out}


def answer_node(state: DemoState):
    route = state.get("route", "dsl")
    reason = state.get("reason", "")
    provider_name = state.get("router_provider", "unknown")
    question = state.get("user_question", "")

    if route == "dsl":
        result = state.get("dsl_result", [])
    elif route == "llm":
        result = state.get("llm_result", [])
    else:
        result = state.get("final_candidates", [])

    candidate_lookup = {c["id"]: c for c in candidates}
    selected_candidates = [
        {
            "id": cid,
            "full_name": candidate_lookup.get(cid, {}).get("full_name", ""),
            "current_job_title": candidate_lookup.get(cid, {}).get("current_job_title", ""),
            "experience_text": candidate_lookup.get(cid, {}).get("experience_text", ""),
            "skills_text": candidate_lookup.get(cid, {}).get("skills_text", ""),
            "summary_text": candidate_lookup.get(cid, {}).get("summary_text", ""),
        }
        for cid in result
    ]

    final_payload = {
        "route": route,
        "router_provider": provider_name,
        "reason": reason,
        "candidate_ids": result,
        "selected_candidates": selected_candidates,
    }

    system_prompt = (
        "You are a senior recruitment assistant. "
        "Write a smooth, concise, user-facing Vietnamese answer. "
        "Only use provided CV facts and do not hallucinate."
    )
    user_prompt = (
        f"Cau hoi user: {question}\n\n"
        f"Thong tin ket qua tu graph (JSON):\n"
        f"{json.dumps(final_payload, ensure_ascii=False, indent=2)}\n\n"
        "Hay tra loi user that tu nhien, ro rang, co giai thich ngan gon vi sao chon ung vien."
    )

    try:
        provider = LLMProvider()
        llm_resp = provider.generate(prompt=user_prompt, system_prompt=system_prompt)
        final_answer = (llm_resp.text or "").strip()
        if not final_answer:
            raise ValueError("empty final answer")
    except Exception:
        final_answer = (
            f"Ung vien de xuat: {', '.join(result) if result else 'khong co'}. "
            f"Ly do: {reason}"
        )

    return {"final_payload": final_payload, "answer": final_answer}


builder = StateGraph(DemoState)

builder.add_node("route_node", route_node)
builder.add_node("dsl", dsl_node)
builder.add_node("llm", llm_node)
builder.add_node("hybrid", hybrid_exec_node)
builder.add_node("answer", answer_node)

builder.add_edge(START, "route_node")
builder.add_conditional_edges(
    "route_node",
    route_selector,
    {
        "dsl": "dsl",
        "llm": "llm",
        "hybrid": "hybrid",
    },
)

builder.add_edge("dsl", "answer")
builder.add_edge("llm", "answer")
builder.add_edge("hybrid", "answer")
builder.add_edge("answer", END)

graph = builder.compile()

input_state: DemoState = {
    "user_question": "Chỉ sử dụng thông tin trong CV, hãy cho tôi biết ứng viên nào có kinh nghiệm marketing và kỹ năng phù hợp nhất?",
}

output = graph.invoke(input_state)
print(json.dumps(output, ensure_ascii=False, indent=2))

{
  "user_question": "Chỉ sử dụng thông tin trong CV, hãy cho tôi biết ứng viên nào có kinh nghiệm marketing và kỹ năng phù hợp nhất?",
  "route": "dsl",
  "reason": "The user requests a selection based on specific criteria (marketing experience and suitable skills) that can be addressed through structured filtering and ranking of CV data.",
  "router_raw": "{\"route\":\"dsl\",\"reason\":\"The user requests a selection based on specific criteria (marketing experience and suitable skills) that can be addressed through structured filtering and ranking of CV data.\"}",
  "router_provider": "shopaikey",
  "dsl_result": [
    "cv-001",
    "cv-002",
    "cv-003"
  ],
  "final_payload": {
    "route": "dsl",
    "router_provider": "shopaikey",
    "reason": "The user requests a selection based on specific criteria (marketing experience and suitable skills) that can be addressed through structured filtering and ranking of CV data.",
    "candidate_ids": [
      "cv-001",
      "cv-002",
     